In [5]:
!pip freeze | grep scikit-learn

scikit-learn==1.5.0


In [6]:
!python -V

Python 3.10.11


In [15]:
import pickle
import pandas as pd
import numpy as np

In [8]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [9]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [10]:
df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')

In [11]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [13]:
y_pred.std()

np.float64(6.247488852238703)

In [17]:
# Create artificial ride_id column
year = 2023
month = 3
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [18]:
# Create results dataframe with ride_id and predictions
df_result = pd.DataFrame({
    'ride_id': df['ride_id'],
    'predicted_duration': y_pred
})

In [19]:
print(f"\nResults dataframe shape: {df_result.shape}")
print(f"Results dataframe columns: {list(df_result.columns)}")
print(f"Data types:\n{df_result.dtypes}")


Results dataframe shape: (3316216, 2)
Results dataframe columns: ['ride_id', 'predicted_duration']
Data types:
ride_id                object
predicted_duration    float64
dtype: object


In [20]:
# Save as parquet file
output_file = 'predictions_2023_03.parquet'
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [21]:
# Check the file size
import os
file_size = os.path.getsize(output_file)
file_size_mb = file_size / (1024 * 1024)

In [22]:
print(f"\nOutput file saved: {output_file}")
print(f"File size: {file_size} bytes")
print(f"File size: {file_size_mb:.2f} MB")


Output file saved: predictions_2023_03.parquet
File size: 68640926 bytes
File size: 65.46 MB


In [23]:
# Display first few rows of the result
print(f"\nFirst 5 rows of results:")
print(df_result.head())


First 5 rows of results:
     ride_id  predicted_duration
0  2023/03_0           16.245906
1  2023/03_1           26.134796
2  2023/03_2           11.884264
3  2023/03_3           11.997720
4  2023/03_4           10.234486
